In [41]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3
import pandas as pd

In [42]:
conn = sqlite3.connect("../../db/dev.sqlite")

In [43]:
df = pd.read_csv("../raw-data/worldbank_data.csv", encoding="latin1", engine="python", na_values = '..')

In [44]:
df.describe()
metadata = pd.read_csv('../raw-data/worldbank_metadata.csv', encoding = 'latin1')

In [45]:
years_cols = [col for col in df.columns if col[:4].isdigit()]

df_long = df.melt(
    id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'],
    value_vars=years_cols,
    var_name = 'year',
    value_name = 'value'
)

In [46]:
df_long["year"] = df_long["year"].str.extract(r"(\d{4})")

In [47]:
df_wide = df_long.pivot_table(
    index = ['Country Name', 'Country Code', 'year'],
    columns = 'Series Name',
    values = 'value'
).reset_index()

In [48]:
df_wide = df_wide.rename(columns={
    'Country Name' : 'country',
    'Country Code' : 'country_code',
    'GDP growth (annual %)' : 'gdp_growth',
    'Inflation, consumer prices (annual %)': 'inflation',
    'Life expectancy at birth, total (years)': 'life_expectancy',
    'Poverty gap at $3.00 a day (2021 PPP) (%)' : 'poverty'
})

In [49]:
df_wide.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17178 entries, 0 to 17177
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country          17178 non-null  object 
 1   country_code     17178 non-null  object 
 2   year             17178 non-null  object 
 3   gdp_growth       14133 non-null  float64
 4   inflation        11295 non-null  float64
 5   life_expectancy  16926 non-null  float64
 6   poverty          2970 non-null   float64
dtypes: float64(4), object(3)
memory usage: 939.5+ KB


In [50]:
df_wide['year'] = df_wide['year'].astype(int)

In [51]:
df_wide = df_wide[df_wide['year'] >= 1990]

In [52]:
df_wide['country'].unique()

array(['Afghanistan', 'Africa Eastern and Southern',
       'Africa Western and Central', 'Albania', 'Algeria',
       'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda',
       'Arab World', 'Argentina', 'Armenia', 'Aruba', 'Australia',
       'Austria', 'Azerbaijan', 'Bahamas, The', 'Bahrain', 'Bangladesh',
       'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda',
       'Bhutan', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana',
       'Brazil', 'British Virgin Islands', 'Brunei Darussalam',
       'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia',
       'Cameroon', 'Canada', 'Caribbean small states', 'Cayman Islands',
       'Central African Republic', 'Central Europe and the Baltics',
       'Chad', 'Channel Islands', 'Chile', 'China', 'Colombia', 'Comoros',
       'Congo, Dem. Rep.', 'Congo, Rep.', 'Costa Rica', "Cote d'Ivoire",
       'Croatia', 'Cuba', 'Curacao', 'Cyprus', 'Czechia', 'Denmark',
       'Djibouti', 'Dominica', 'Dominican Repub

In [53]:
not_countries = [
"Africa Eastern and Southern",
"Africa Western and Central",
"Arab World",
"Caribbean small states",
"Central Europe and the Baltics",
"Early-demographic dividend",
"East Asia & Pacific",
"East Asia & Pacific (IDA & IBRD countries)",
"East Asia & Pacific (excluding high income)",
"Euro area",
"Europe & Central Asia",
"Europe & Central Asia (IDA & IBRD countries)",
"Europe & Central Asia (excluding high income)",
"European Union",
"Fragile and conflict affected situations",
"Heavily indebted poor countries (HIPC)",
"High income",
"IBRD only",
"IDA & IBRD total",
"IDA blend",
"IDA only",
"IDA total",
"Late-demographic dividend",
"Latin America & Caribbean",
"Latin America & Caribbean (excluding high income)",
"Latin America & the Caribbean (IDA & IBRD countries)",
"Least developed countries: UN classification",
"Low & middle income",
"Low income",
"Lower middle income",
"Middle East, North Africa, Afghanistan & Pakistan",
"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",
"Middle East, North Africa, Afghanistan & Pakistan (excluding high income)",
"Middle income",
"North America",
"OECD members",
"Other small states",
"Pacific island small states",
"Post-demographic dividend",
"Pre-demographic dividend",
"Small states",
"South Asia",
"South Asia (IDA & IBRD)",
"Sub-Saharan Africa",
"Sub-Saharan Africa (IDA & IBRD countries)",
"Sub-Saharan Africa (excluding high income)",
"Upper middle income",
"World"
]

df_wide = df_wide[~df_wide['country'].isin(not_countries)]

In [54]:
df_wide['country'].nunique()

217

In [55]:
metadata = pd.read_csv('../raw-data/worldbank_metadata.csv', encoding = 'latin1')

In [56]:
df_wide = df_wide.merge(metadata, left_on = 'country_code', right_on='Country Name', how = 'left')

In [57]:
df_wide = df_wide.rename(columns= {
    'Series Code': 'region',
    'Series Name': 'income'
})

In [58]:
df_wide = df_wide[['country', 'country_code', 'region', 'income', 'year', 'gdp_growth','inflation', 'life_expectancy', 'poverty']]

In [59]:
df_wide

,country,country_code,region,income,year,gdp_growth,inflation,life_expectancy,poverty
0,Afghanistan,AFG,Middle East & North Africa,Low income,1990,NaN,NaN,45.118,NaN
1,Afghanistan,AFG,Middle East & North Africa,Low income,1991,NaN,NaN,45.521,NaN
2,Afghanistan,AFG,Middle East & North Africa,Low income,1992,NaN,NaN,46.569,NaN
3,Afghanistan,AFG,Middle East & North Africa,Low income,1993,NaN,NaN,51.021,NaN
4,Afghanistan,AFG,Middle East & North Africa,Low income,1994,NaN,NaN,50.969,NaN
...,...,...,...,...,...,...,...,...,...
7573,Zimbabwe,ZWE,Sub-Saharan Africa,Lower middle income,2020,-7.816951,557.201817,61.530,NaN
7574,Zimbabwe,ZWE,Sub-Saharan Africa,Lower middle income,2021,8.468017,98.546105,60.135,NaN
7575,Zimbabwe,ZWE,Sub-Saharan Africa,Lower middle income,2022,6.139263,104.705171,62.360,NaN
7576,Zimbabwe,ZWE,Sub-Saharan Africa,Lower middle income,2023,5.350869,NaN,62.775,NaN


In [60]:
df_wide.to_csv("../raw-data/economic_data_clean.csv", index = False)

In [61]:
df_latest = df_wide[df_wide['year']==2022]

In [62]:
df_latest

,country,country_code,region,income,year,gdp_growth,inflation,life_expectancy,poverty
32,Afghanistan,AFG,Middle East & North Africa,Low income,2022,-6.240172,13.712102,65.617000,NaN
67,Albania,ALB,Europe & Central Asia,Upper middle income,2022,4.826801,6.725203,78.769000,NaN
102,Algeria,DZA,Middle East & North Africa,Upper middle income,2022,3.600000,9.265516,76.129000,NaN
137,American Samoa,ASM,East Asia & Pacific,High income,2022,1.735016,NaN,72.752000,NaN
171,Andorra,AND,Europe & Central Asia,High income,2022,9.564612,NaN,84.016000,NaN
...,...,...,...,...,...,...,...,...,...
7437,Virgin Islands (U.S.),VIR,Latin America & Caribbean,High income,2022,-1.311232,NaN,80.319512,NaN
7471,West Bank and Gaza,PSE,Middle East & North Africa,Lower middle income,2022,4.082760,3.741224,76.662000,NaN
7506,"Yemen, Rep.",YEM,Middle East & North Africa,Low income,2022,NaN,NaN,67.952000,NaN
7540,Zambia,ZMB,Sub-Saharan Africa,Lower middle income,2022,5.211224,10.993204,65.279000,39.8


In [63]:
df_latest.to_csv('../raw-data/economic_latest.csv', index = False)

In [64]:
df_wide.to_sql(
    "economic_history",
    conn,
    if_exists="append",
    index=False
)

7578

In [65]:
df_latest.to_sql(
    "economic_latest",
    conn,
    if_exists="append",
    index=False
)

217

In [66]:
conn.close()